### Kernel: Python

## Treinamento da rede e avaliacao para os dados de teste (covariaveis selecionadas)

In [14]:
# Configuracoes do ambiente

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 2003
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.0025

np.random.seed(SEED)
torch.manual_seed(SEED)

In [15]:
# Carregamento dos dados

entrada_rede = pd.read_csv("../dados/entrada/entrada_rede.csv")

# Covariaveis selecionadas pelo estudo anterior
COX_COVARS = [
    "stage_paper",
    "prior_diag_trat_paper",
    "hormonioterapia_paper",
    "educ_paper",
    "quimioterapia_paper",
    "macroregiao_paper",
]

REQUIRED_RAW_COLS = [
    "rhc_estadiamento_clinico",
    "rhc_diagnostico_e_tratamentos_anteriores",
    "rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia",
    "rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia",
    "instrucao",
    "macroregiao",
]

def prepara_covariaveis_selecionadas(df):
    df = df.copy()

    # Se as colunas brutas estiverem presentes, faz a recodificacao
    if all(col in df.columns for col in REQUIRED_RAW_COLS):
        stage_raw = df["rhc_estadiamento_clinico"]
        df["stage_paper"] = stage_raw.map({"I e II": "I/II", "III e IV": "III/IV"})

        prior_raw = df["rhc_diagnostico_e_tratamentos_anteriores"]
        df["prior_diag_trat_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
        df.loc[prior_raw == "Com diag./Com trat.", "prior_diag_trat_paper"] = "Com diag./Com trat."
        df.loc[prior_raw.isin(["Com diag./Sem trat.", "Sem diag./Sem trat."]), "prior_diag_trat_paper"] = "Outros"

        horm_raw = df["rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia"]
        df["hormonioterapia_paper"] = horm_raw.map({"Sim": "Sim", "Nao": "Nao", "Não": "Nao"})

        quimio_raw = df["rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia"]
        df["quimioterapia_paper"] = quimio_raw.map({"Sim": "Sim", "Nao": "Nao", "Não": "Nao"})

        educ_raw = df["instrucao"]
        df["educ_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
        df.loc[educ_raw == 0, "educ_paper"] = "None"
        df.loc[educ_raw.isin([1, 2]), "educ_paper"] = "Primary/Secondary+"

        macro_raw = df["macroregiao"]
        df["macroregiao_paper"] = pd.Series(pd.NA, index=df.index, dtype="string")
        df.loc[macro_raw == 2, "macroregiao_paper"] = "2"
        df.loc[macro_raw.isin([1, 3]), "macroregiao_paper"] = "1/3"
    else:
        missing_covs = [c for c in COX_COVARS if c not in df.columns]
        if missing_covs:
            raise ValueError(f"Colunas ausentes para covariaveis selecionadas: {missing_covs}")

    # Mantem apenas linhas com tempo valido e covariaveis completas
    df = df[df["tempo"] > 0]
    df = df.dropna(subset=COX_COVARS)
    return df

dados_modelo = prepara_covariaveis_selecionadas(entrada_rede)

# Split treino/teste
dados_treino = dados_modelo.loc[dados_modelo["conjunto"] == "treino"].copy()
dados_teste = dados_modelo.loc[dados_modelo["conjunto"] == "teste"].copy()

# Pseudo apenas para treino
dados_treino = dados_treino[dados_treino["pseudo"].notna()].copy()

dados_treino.head()

,id_paciente,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,...,diff_data_tratamento,tempo,pseudo,conjunto,stage_paper,prior_diag_trat_paper,hormonioterapia_paper,quimioterapia_paper,educ_paper,macroregiao_paper
0,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,42.0,2.0,1.000004,treino,III/IV,Outros,Nao,Sim,None,1/3
1,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,42.0,7.0,1.000116,treino,III/IV,Outros,Nao,Sim,None,1/3
2,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,42.0,11.0,1.000329,treino,III/IV,Outros,Nao,Sim,None,1/3
3,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,42.0,15.0,1.000705,treino,III/IV,Outros,Nao,Sim,None,1/3
4,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,...,42.0,20.0,-0.038298,treino,III/IV,Outros,Nao,Sim,None,1/3


In [16]:
# Sequencia de tempos para avaliacao da rede nos dados de teste

tempos_avaliacao_teste = np.arange(1, 132, 1)

## Treinamento e Previsoes para os dados de teste das Redes Neurais

In [17]:
# Definicao da estrutura da rede neural

class rede_sobrevivencia(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.Tanh(),
            nn.Linear(input_dim, 2 * input_dim),
            nn.Tanh(),
            nn.Linear(2 * input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

## Rede Neural A: one-hot encoding

In [18]:
def normaliza_coluna_numerica(serie, stats_coluna):
    valores = pd.to_numeric(serie, errors="coerce").fillna(stats_coluna["mediana"])
    return (valores - stats_coluna["media"]) / (stats_coluna["desvio"] + 1e-8)


covariaveis = COX_COVARS
covariaveis_numericas = [col for col in covariaveis if pd.api.types.is_numeric_dtype(dados_treino[col])]
covariaveis_categoricas = [col for col in covariaveis if col not in covariaveis_numericas]

estatisticas_numericas = {}
for col in covariaveis_numericas:
    serie = pd.to_numeric(dados_treino[col], errors="coerce")
    mediana = float(serie.median()) if serie.notna().any() else 0.0
    serie_imp = serie.fillna(mediana)
    estatisticas_numericas[col] = {
        "mediana": mediana,
        "media": float(serie_imp.mean()),
        "desvio": float(serie_imp.std(ddof=0)),
    }

mapeamento_categorias = {}
for col in covariaveis_categoricas:
    mapeamento_categorias[col] = sorted(dados_treino[col].fillna("NA").astype(str).unique().tolist())


def gera_matriz_design_one_hot(dados, tempos):
    dados_ref = dados.reset_index(drop=True).copy()

    # Preenche covariaveis ausentes em bases de predicao
    missing_covs = [col for col in covariaveis if col not in dados_ref.columns]
    for col in missing_covs:
        dados_ref[col] = pd.NA

    base_num = pd.DataFrame(index=dados_ref.index)
    for col in covariaveis_numericas:
        base_num[col] = normaliza_coluna_numerica(dados_ref[col], estatisticas_numericas[col])

    if len(covariaveis_categoricas) > 0:
        base_cat = dados_ref[covariaveis_categoricas].copy()
        for col in covariaveis_categoricas:
            base_cat[col] = pd.Categorical(
                base_cat[col].fillna("NA").astype(str),
                categories=mapeamento_categorias[col],
                ordered=False,
            )
        base_cat = pd.get_dummies(base_cat, prefix=covariaveis_categoricas, dtype=float)
        base_cat = base_cat.reset_index(drop=True)
    else:
        base_cat = pd.DataFrame(index=dados_ref.index)

    tcat = pd.Categorical(
        pd.to_numeric(dados_ref["tempo"], errors="coerce").astype(float),
        categories=tempos,
        ordered=True,
    )
    t_oh = pd.get_dummies(tcat, prefix="tempo", dtype=float).reset_index(drop=True)

    return pd.concat([base_num, base_cat, t_oh], axis=1)


time_levels = np.sort(dados_treino["tempo"].astype(float).unique())
x_treino_onehot = gera_matriz_design_one_hot(dados_treino, time_levels)
y_treino_onehot = dados_treino["pseudo"].astype(float).values.reshape(-1, 1)

X_train_onehot = torch.tensor(x_treino_onehot.values.astype(np.float32))
y_train_onehot = torch.tensor(y_treino_onehot.astype(np.float32))

train_loader_onehot = DataLoader(
    TensorDataset(X_train_onehot, y_train_onehot),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
 )

print(f"dimensao X_train: {tuple(X_train_onehot.shape)}")
print(f"dimensao y_train: {tuple(y_train_onehot.shape)}")

dimensao X_train: (48610, 22)
dimensao y_train: (48610, 1)


In [19]:
# Definicao do modelo, criterio e otimizador para a rede one-hot
model_onehot = rede_sobrevivencia(input_dim=X_train_onehot.shape[1])
criterion_onehot = nn.MSELoss()
optimizer_onehot = optim.Adam(model_onehot.parameters(), lr=LEARNING_RATE)

In [20]:
# Treinamento da rede one-hot

for epoch in range(EPOCHS):
    model_onehot.train()
    running_loss = 0.0

    for xb, yb in train_loader_onehot:
        optimizer_onehot.zero_grad()
        pred = model_onehot(xb)
        loss = criterion_onehot(pred, yb)
        loss.backward()
        optimizer_onehot.step()
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoca {epoch + 1:4d}/{EPOCHS} | mse={(running_loss / len(train_loader_onehot)):.6f}")

Epoca   10/100 | mse=0.063107
Epoca   20/100 | mse=0.062835
Epoca   30/100 | mse=0.062687
Epoca   40/100 | mse=0.062676
Epoca   50/100 | mse=0.062596
Epoca   60/100 | mse=0.062601
Epoca   70/100 | mse=0.062545
Epoca   80/100 | mse=0.062506
Epoca   90/100 | mse=0.062521
Epoca  100/100 | mse=0.062525


In [21]:
# Pasta de saida das predicoes
dir_saida = Path("../dados/saida")

# IDs unicos de teste
dados_ids_teste = dados_teste[["id_paciente"] + covariaveis].drop_duplicates().sort_values("id_paciente")

# Base com os tempos de avaliacao do teste
linhas_teste_evento = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_evento in tempos_avaliacao_teste:
        linha = {
            "id_paciente": int(linha_individuo["id_paciente"]),
            "tempo": float(tempo_evento),
        }
        for col in covariaveis:
            linha[col] = linha_individuo[col]
        linhas_teste_evento.append(linha)
dados_teste_evento = pd.DataFrame(
    linhas_teste_evento,
    columns=["id_paciente", "tempo"] + covariaveis,
 )

# Predicoes Rede one-hot
linhas_grade_A = []
for _, linha_individuo in dados_ids_teste.iterrows():
    for tempo_grade in time_levels:
        linha = {
            "id_paciente": int(linha_individuo["id_paciente"]),
            "tempo": float(tempo_grade),
        }
        for col in covariaveis:
            linha[col] = linha_individuo[col]
        linhas_grade_A.append(linha)

grade_predicao_A = pd.DataFrame(
    linhas_grade_A,
    columns=["id_paciente", "tempo"] + covariaveis,
 )
x_pred_A = gera_matriz_design_one_hot(grade_predicao_A, time_levels)
x_pred_A = x_pred_A.reindex(columns=x_treino_onehot.columns, fill_value=0.0)

# Predicao da Rede one-hot na grade de treino
model_onehot.eval()
with torch.no_grad():
    grade_predicao_A["pred_s"] = model_onehot(torch.tensor(x_pred_A.values.astype(np.float32))).cpu().numpy().reshape(-1)

grade_predicao_A = grade_predicao_A.sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

# Predicao da Rede one-hot nos tempos de avaliacao do teste (via interpolacao)
linhas_evento_A = []
for id_individuo, predicoes_individuo in grade_predicao_A.groupby("id_paciente"):
    predicoes_individuo = predicoes_individuo.sort_values("tempo")
    predicoes_evento = np.interp(
        tempos_avaliacao_teste,
        predicoes_individuo["tempo"].values,
        predicoes_individuo["pred_s"].values,
    )
    for tempo_evento, valor_predito in zip(tempos_avaliacao_teste, predicoes_evento):
        linhas_evento_A.append({
            "id_paciente": int(id_individuo),
            "tempo": float(tempo_evento),
            "pred_s": float(valor_predito),
        })

pred_event_A = pd.DataFrame(
    linhas_evento_A,
    columns=["id_paciente", "tempo", "pred_s"],
).sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

arquivo_saida_A = dir_saida / "predicao-rede-A-onehot-covsel.csv"
pred_event_A_out = pred_event_A.rename(columns={"id_paciente": "ID", "tempo": "TIME", "pred_s": "PRED_S"})
pred_event_A_out.to_csv(arquivo_saida_A, index=False)

## Rede Neural B: tempo continuo

In [22]:
# Preparacao dos dados para a rede usando tempo continuo normalizado e categoricas one-hot

tempo_treino = pd.to_numeric(dados_treino["tempo"], errors="coerce").astype(float)

stats_tempo = {
    "mediana": float(tempo_treino.median()),
    "media": float(tempo_treino.mean()),
    "desvio": float(tempo_treino.std(ddof=0)),
}


def gera_matriz_design_tempo_continuo(dados):
    dados_ref = dados.reset_index(drop=True).copy()

    # Preenche covariaveis ausentes em bases de predicao
    missing_covs = [col for col in covariaveis if col not in dados_ref.columns]
    for col in missing_covs:
        dados_ref[col] = pd.NA

    base_num = pd.DataFrame(index=dados_ref.index)

    for col in covariaveis_numericas:
        base_num[col] = normaliza_coluna_numerica(
            dados_ref[col],
            estatisticas_numericas[col]
        )

    base_num["tempo"] = normaliza_coluna_numerica(
        dados_ref["tempo"],
        stats_tempo
    )

    if len(covariaveis_categoricas) > 0:
        base_cat = dados_ref[covariaveis_categoricas].copy()

        for col in covariaveis_categoricas:
            base_cat[col] = pd.Categorical(
                base_cat[col].fillna("NA").astype(str),
                categories=mapeamento_categorias[col],
                ordered=False,
            )

        base_cat = pd.get_dummies(
            base_cat,
            prefix=covariaveis_categoricas,
            dtype=float
        ).reset_index(drop=True)
    else:
        base_cat = pd.DataFrame(index=dados_ref.index)

    return pd.concat([base_num, base_cat], axis=1)


x_treino_B = gera_matriz_design_tempo_continuo(dados_treino)
y_treino_B = dados_treino["pseudo"].astype(float).values.reshape(-1, 1)

X_train_B = torch.tensor(x_treino_B.values.astype(np.float32))
y_train_B = torch.tensor(y_treino_B.astype(np.float32))

train_loader_B = DataLoader(
    TensorDataset(X_train_B, y_train_B),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
 )

In [23]:
# Definicao do modelo, criterio e otimizador para a rede com tempos originais

model_B = rede_sobrevivencia(input_dim=X_train_B.shape[1])
criterion_B = nn.MSELoss()
optimizer_B = optim.Adam(model_B.parameters(), lr=LEARNING_RATE)

In [24]:
# Treinamento da rede B com tempos originais

for epoch_B in range(EPOCHS):
    model_B.train()
    running_loss_B = 0.0

    for x_batch_B, y_batch_B in train_loader_B:
        optimizer_B.zero_grad()
        pred_B = model_B(x_batch_B)
        loss_B = criterion_B(pred_B, y_batch_B)
        loss_B.backward()
        optimizer_B.step()
        running_loss_B += loss_B.item()

    if (epoch_B + 1) % 10 == 0:
        print(f"Epoca {epoch_B + 1:4d}/{EPOCHS} | mse_B={(running_loss_B / len(train_loader_B)):.6f}")

Epoca   10/100 | mse_B=0.062970
Epoca   20/100 | mse_B=0.062761
Epoca   30/100 | mse_B=0.062590
Epoca   40/100 | mse_B=0.062623
Epoca   50/100 | mse_B=0.062613
Epoca   60/100 | mse_B=0.062582
Epoca   70/100 | mse_B=0.062517
Epoca   80/100 | mse_B=0.062469
Epoca   90/100 | mse_B=0.062479
Epoca  100/100 | mse_B=0.062478


In [25]:
# Predicao da Rede B nos tempos de avaliacao do teste
x_evento_B = gera_matriz_design_tempo_continuo(dados_teste_evento)
x_evento_B = x_evento_B.reindex(columns=x_treino_B.columns, fill_value=0.0)

model_B.eval()
with torch.no_grad():
    pred_s_B = model_B(torch.tensor(x_evento_B.values.astype(np.float32))).cpu().numpy().reshape(-1)

pred_event_B = dados_teste_evento[["id_paciente", "tempo"]].copy()
pred_event_B["pred_s"] = pred_s_B
pred_event_B = pred_event_B.sort_values(["id_paciente", "tempo"]).reset_index(drop=True)

arquivo_saida_B = dir_saida / "predicao-rede-B-tempo-continuo-covsel.csv"
pred_event_B_out = pred_event_B.rename(columns={"id_paciente": "ID", "tempo": "TIME", "pred_s": "PRED_S"})
pred_event_B_out.to_csv(arquivo_saida_B, index=False)